# Реализация статистических тестов

## Подключим библиотеки для работы

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

## Подготовим данные для начала

Перед проверкой гипотез лучше нам еще раз убедиться, что с данными все в порядке, нет дубликатов, пропусков, что названия колонок корректны

In [4]:
data = pd.read_csv('data.csv', sep=';')

In [5]:
data.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [6]:
data.shape

(4424, 37)

#### Проверка колонок

In [8]:
data.columns

Index(['Marital status', 'Application mode', 'Application order', 'Course',
       'Daytime/evening attendance\t', 'Previous qualification',
       'Previous qualification (grade)', 'Nacionality',
       'Mother's qualification', 'Father's qualification',
       'Mother's occupation', 'Father's occupation', 'Admission grade',
       'Displaced', 'Educational special needs', 'Debtor',
       'Tuition fees up to date', 'Gender', 'Scholarship holder',
       'Age at enrollment', 'International',
       'Curricular units 1st sem (credited)',
       'Curricular units 1st sem (enrolled)',
       'Curricular units 1st sem (evaluations)',
       'Curricular units 1st sem (approved)',
       'Curricular units 1st sem (grade)',
       'Curricular units 1st sem (without evaluations)',
       'Curricular units 2nd sem (credited)',
       'Curricular units 2nd sem (enrolled)',
       'Curricular units 2nd sem (evaluations)',
       'Curricular units 2nd sem (approved)',
       'Curricular units 2nd

In [9]:
data.columns = data.columns.str.strip()

In [10]:
data.columns

Index(['Marital status', 'Application mode', 'Application order', 'Course',
       'Daytime/evening attendance', 'Previous qualification',
       'Previous qualification (grade)', 'Nacionality',
       'Mother's qualification', 'Father's qualification',
       'Mother's occupation', 'Father's occupation', 'Admission grade',
       'Displaced', 'Educational special needs', 'Debtor',
       'Tuition fees up to date', 'Gender', 'Scholarship holder',
       'Age at enrollment', 'International',
       'Curricular units 1st sem (credited)',
       'Curricular units 1st sem (enrolled)',
       'Curricular units 1st sem (evaluations)',
       'Curricular units 1st sem (approved)',
       'Curricular units 1st sem (grade)',
       'Curricular units 1st sem (without evaluations)',
       'Curricular units 2nd sem (credited)',
       'Curricular units 2nd sem (enrolled)',
       'Curricular units 2nd sem (evaluations)',
       'Curricular units 2nd sem (approved)',
       'Curricular units 2nd s

#### Дубликаты и пропуски

In [11]:
data.duplicated().sum()

np.int64(0)

In [13]:
data.isna().sum().sum()

np.int64(0)

In [16]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualification                          4424

В датасете 4424 наблюдения и 37 колонок. Полных дубликатов и пропущенных значений не обнаружено.  
Названия колонок были очищены от лишних пробелов

#### Посмотрим на целевую переменную

Целевая переменная `Target` имеет три категории:
- `Dropout` = студент отчислился
- `Graduate` = студент окончил обучение
- `Enrolled` = студент все еще обучается

Посмотрим на распределение целевой переменной

In [14]:
data['Target'].value_counts()

Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

In [15]:
data['Target'].value_counts(normalize=True)

Target
Graduate    0.499322
Dropout     0.321203
Enrolled    0.179476
Name: proportion, dtype: float64

Почти половина студентов относится к категории `Graduate`, около трети к `Dropout` и около 18% к `Enrolled`. Важно учитывать, что `Enrolled` это лишь промежуточный статус. Поэтому в дальнейших статистических тестах нужно будет это понимать, что статус по факту может быть либо финальным, либо промежуточным

#### Быстрая описательная статистика

In [18]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
Marital status,4424.0,1.178571,0.605747,1.00,1.00,1.000000,1.000000,6.000000
Application mode,4424.0,18.669078,17.484682,1.00,1.00,17.000000,39.000000,57.000000
Application order,4424.0,1.727848,1.313793,0.00,1.00,1.000000,2.000000,9.000000
Course,4424.0,8856.642631,2063.566416,33.00,9085.00,9238.000000,9556.000000,9991.000000
Daytime/evening attendance,4424.0,0.890823,0.311897,0.00,1.00,1.000000,1.000000,1.000000
Previous qualification,4424.0,4.577758,10.216592,1.00,1.00,1.000000,1.000000,43.000000
Previous qualification (grade),4424.0,132.613314,13.188332,95.00,125.00,133.100000,140.000000,190.000000
Nacionality,4424.0,1.873192,6.914514,1.00,1.00,1.000000,1.000000,109.000000
Mother's qualification,4424.0,19.561935,15.603186,1.00,2.00,19.000000,37.000000,44.000000
Father's qualification,4424.0,22.275316,15.343108,1.00,3.00,19.000000,37.000000,44.000000


Важно учитывать, что не все числовые признаки являются количественными. Часть колонок представляет собой закодированные категории, например `Course`, `Application mode`, `Marital status`, `Mother's qualification` ...

Поэтому в дальнейшей части тип переменной и подходящий статистический тест будут определяться отдельно для каждой гипотезы

## Проверка статистических гипотез

### Гипотеза 1. Академический прогресс в 1 семестре и риск отчисления

Изначально можно было бы использовать например только абсолютное количество сданных дисциплин `Curricular units 1st sem (approved)`. Однако такой подход не полностью корректен, потому что студенты могли быть записаны на разное количество дисциплин. Например студент мог сдать 3 дисциплины из 3, и это не будет означать низкую успеваемость. Поэтому вместо абсолютного количества сданных дисциплин будем использовать относительный показатель, расчитанный как approval_rate_1sem = approved_1sem / enrolled_1sem

То есть будем смотреть, какую долю дисциплин студент сдал от числа дисциплин, на которые он был записан в первом семестре

В качестве индикатора низкого академического прогресса будем использовать условие `approval_rate_1sem < 0.5`. То есть студент сдал менее 50% дисциплин, на которые был записан в первом семестре.

Лучше нам также разделить целевую переменную на две группы для анализа в этот раз:
- `1` - студент отчислился (`Dropout`)
- `0` - студент не отчислился (`Graduate` или `Enrolled`). Категория `Enrolled` здесь включается в группу, потому что в этой гипотезе мы проверяем именно факт попадания студента в группу `Dropout`

**Формулировка гипотезы:**  
Студенты, которые в первом семестре сдали менее 50% дисциплин, имеют статистически значимо более высокую долю отчислений по сравнению со студентами, сдавшими 50% и более дисциплин

В датасете есть разные признаки, описывающие академическую ситуацию студента в первом семестре:

|Признак|Смысл|
|---|---|
|`Curricular units 1st sem (credited)`|количество дисциплин, зачтенных студенту|
|`Curricular units 1st sem (enrolled)`|количество дисциплин, на которые студент был записан|
|`Curricular units 1st sem (evaluations)`|количество оцениваний в 1 семестре|
|`Curricular units 1st sem (approved)`|количество сданных дисциплин|
|`Curricular units 1st sem (grade)`|средняя оценка за 1 семестр|

Тогда наш показатель `approval_rate_1sem = Curricular units 1st sem (approved) / Curricular units 1st sem (enrolled)`

Нужно проверить, есть ли строки с 0 в признаке `Curricular units 1st sem (enrolled)`, т.к делить на 0 нельзя

In [25]:
data[data['Curricular units 1st sem (enrolled)'] == 0].head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.0,0,10.8,1.4,1.74,Dropout
20,1,1,3,171,1,1,122.0,1,1,1,...,0,0,0,0,0.0,0,11.1,0.6,2.02,Graduate
59,1,1,3,171,1,1,125.0,1,38,37,...,0,0,0,0,0.0,0,7.6,2.6,0.32,Enrolled
62,1,17,3,171,1,1,133.0,1,1,37,...,0,0,0,0,0.0,0,10.8,1.4,1.74,Enrolled
66,1,1,3,171,1,1,139.0,1,19,19,...,0,0,0,0,0.0,0,10.8,1.4,1.74,Graduate


In [29]:
data[data['Curricular units 1st sem (enrolled)'] == 0].shape[0]

180

In [49]:
data[data['Curricular units 1st sem (enrolled)'] == 0].shape[0]/data.shape[0] * 100

4.06871609403255

Таких студентов около 180 (4% немного, но они есть). Это надо учесть в дальнейшем. Студенты с `enrolled = 0` не будут использоваться в этой гипотезе, потому что для них невозможно корректно рассчитать долю сданных дисциплин

Подготовим данные для гипотезы

In [43]:
data_h1 = data[['Target', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (enrolled)']].copy()

In [44]:
data_h1.head()

,Target,Curricular units 1st sem (approved),Curricular units 1st sem (enrolled)
0,Dropout,0,0
1,Graduate,6,6
2,Dropout,0,6
3,Graduate,6,6
4,Graduate,5,6


In [45]:
data_h1.info()

<class 'pandas.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 3 columns):
 #   Column                               Non-Null Count  Dtype
---  ------                               --------------  -----
 0   Target                               4424 non-null   str  
 1   Curricular units 1st sem (approved)  4424 non-null   int64
 2   Curricular units 1st sem (enrolled)  4424 non-null   int64
dtypes: int64(2), str(1)
memory usage: 103.8 KB


In [46]:
data_h1 = data_h1[data_h1['Curricular units 1st sem (enrolled)'] > 0]

In [47]:
data_h1.info()

<class 'pandas.DataFrame'>
Index: 4244 entries, 1 to 4423
Data columns (total 3 columns):
 #   Column                               Non-Null Count  Dtype
---  ------                               --------------  -----
 0   Target                               4244 non-null   str  
 1   Curricular units 1st sem (approved)  4244 non-null   int64
 2   Curricular units 1st sem (enrolled)  4244 non-null   int64
dtypes: int64(2), str(1)
memory usage: 132.6 KB


Создаем наш показатель

In [50]:
data_h1['approval_rate_1sem'] = data_h1['Curricular units 1st sem (approved)']/data_h1['Curricular units 1st sem (enrolled)']

In [51]:
data_h1.head()

,Target,Curricular units 1st sem (approved),Curricular units 1st sem (enrolled),approval_rate_1sem
1,Graduate,6,6,1.000000
2,Dropout,0,6,0.000000
3,Graduate,6,6,1.000000
4,Graduate,5,6,0.833333
5,Graduate,5,5,1.000000


И как раз добавим бинарный индикатор низкого академического прогресса `low_academic_progress_1sem`. Значение `low_academic_progress_1sem = 1` означает, что студент сдал менее 50% дисциплин, на которые был записан в первом семестре Значение `low_academic_progress_1sem = 0` означает, что студент сдал 50% или более дисциплин

In [54]:
data_h1['low_academic_progress_1sem'] = data_h1['approval_rate_1sem'].apply(lambda x: 1 if x < 0.5 else 0)

In [57]:
target_groups = {'Graduate' : 0, 'Dropout' : 1, 'Enrolled' : 0}
data_h1['is_dropout'] = data_h1['Target'].map(target_groups)

In [58]:
data_h1.head()

,Target,Curricular units 1st sem (approved),Curricular units 1st sem (enrolled),approval_rate_1sem,low_academic_progress_1sem,is_dropout
1,Graduate,6,6,1.000000,0,0
2,Dropout,0,6,0.000000,1,1
3,Graduate,6,6,1.000000,0,0
4,Graduate,5,6,0.833333,0,0
5,Graduate,5,5,1.000000,0,0


Быстренько взглянем на описательную статистику по группам

In [63]:
data_h1.groupby('low_academic_progress_1sem')['is_dropout'].agg(['count', 'sum', 'mean']).rename(columns={'count': 'n_students', 'sum': 'n_dropouts', 'mean': 'dropout_rate'})

,n_students,n_dropouts,dropout_rate
low_academic_progress_1sem,,,
0,3396,625,0.184040
1,848,719,0.847877


Уже видно сильное различие между группами. Среди студентов, сдавших 50% или более дисциплин в первом семестре, доля отчисленных составляет около 18.4%. Среди студентов, сдавших менее 50% дисциплин, доля отчисленных составляет около 84.8%. Но мы понимаем, что описательной статистики недостаточно, чтобы сделать статистический вывод. Далее проверим, является ли различие в долях статистически значимым

#### Формулировка статистических гипотез

Обозначим:

- p_1 = доля отчисленных среди студентов с низким академическим прогрессом в 1 семестре
- p_0 = доля отчисленных среди студентов без низкого академического прогресса

Нулевая гипотеза:
H_0: p_1 = p_0

Альтернативная гипотеза:
H_1: p_1 > p_0

То есть мы проверяем, действительно ли среди студентов, сдавших менее 50% дисциплин в первом семестре, доля отчисленных выше

#### Выбор статистического теста


В этой гипотезе сравниваются две доли:
- доля отчисленных среди студентов с низким академическим прогрессом
- доля отчисленных среди остальных студентов

Обе переменные бинарные:
- `low_academic_progress_1sem`: есть низкий академический прогресс или нет
- `is_dropout`: студент отчислился или не отчислился

Поэтому для проверки гипотезы подходит z-test для двух долей

Используем односторонний тест, потому что альтернативная гипотеза предполагает конкретное направление различия, доля отчислений в группе с низким академическим прогрессом должна быть выше

#### Проверим условия применения теста

Для z-test для двух долей важно, чтобы:

1. наблюдения были независимыми
2. группы были достаточно большими
3. количество "успехов" и "неуспехов" в каждой группе не было слишком малым

Построим таблицу сопряженности

In [66]:
pd.crosstab(data_h1['low_academic_progress_1sem'], data_h1['is_dropout'])

is_dropout,0,1
low_academic_progress_1sem,,
0,2771,625
1,129,719


Видно, что в обеих группах достаточно наблюдений. Поэтому можно использовать нормальную аппроксимацию и применить z-test для двух долей

#### z-test для двух долей

In [68]:
h1_check = data_h1.groupby('low_academic_progress_1sem')['is_dropout'].agg(['count', 'sum', 'mean']).rename(columns={'count': 'n_students', 'sum': 'n_dropouts', 'mean': 'dropout_rate'})
h1_check

,n_students,n_dropouts,dropout_rate
low_academic_progress_1sem,,,
0,3396,625,0.184040
1,848,719,0.847877


In [69]:
h1_check.info()

<class 'pandas.DataFrame'>
Index: 2 entries, 0 to 1
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   n_students    2 non-null      int64  
 1   n_dropouts    2 non-null      int64  
 2   dropout_rate  2 non-null      float64
dtypes: float64(1), int64(2)
memory usage: 64.0 bytes


In [72]:
n_normal = h1_check.loc[0, 'n_students']
n_low = h1_check.loc[1, 'n_students']
n_normal, n_low

(np.int64(3396), np.int64(848))

In [74]:
dropouts_normal = h1_check.loc[0, 'n_dropouts']
dropouts_low = h1_check.loc[1, 'n_dropouts']
dropouts_normal, dropouts_low

(np.int64(625), np.int64(719))

In [75]:
dropouts_low / n_low

np.float64(0.847877358490566)

In [76]:
dropouts_normal / n_normal

np.float64(0.18404004711425206)

In [78]:
from statsmodels.stats.proportion import proportions_ztest

In [79]:
stat, pval = proportions_ztest(count=[dropouts_low, dropouts_normal], nobs=[n_low, n_normal], alternative='larger')

In [80]:
stat

np.float64(37.173416679452814)

In [81]:
pval

np.float64(9.175845802219075e-303)

In [82]:
pval < 0.001

np.True_

Полученное значение z-статистики равно 37.17, а p-value меньше 0.001.

Так как p-value значительно меньше уровня значимости 0.05, мы отвергаем нулевую гипотезу о равенстве долей отчисления в двух группах.

Следовательно, доля отчисленных среди студентов с низким академическим прогрессом в 1 семестре статистически значимо выше, чем среди студентов, сдавших 50% и более дисциплин

- [Z-test: как его проводят и в чем отличие от T-test](https://practicum.yandex.ru/blog/z-test-proverka-gipotez/)
- [Two-sample Proportion Test (Z test) in Python](https://stataiml.com/posts/53_two_sample_prop_test_py/)
- [statsmodels.stats.proportion.proportions_ztest](https://www.statsmodels.org/dev/generated/statsmodels.stats.proportion.proportions_ztest.html)